In [6]:
import numpy as np
import logging
from ase import Atoms

FIELD_AU_TO_VA = 51.4220674763  # a.u. to V/Å

def process_atoms(atoms: Atoms) -> Atoms:
    if 'charge' in atoms.info:
        atoms.info['charge'] = int(atoms.info['charge'])
    else:
        atoms.info['charge'] = 0
    if 'multiplicity' in atoms.info:
        atoms.info['spin'] = int(atoms.info['multiplicity'])
    else:
        atoms.info['spin'] = 1

    if 'multipole_field' in atoms.info:
        try:
            field = atoms.info['multipole_field']
            field_dict = {}
            for item in str(field).split(","):
                if item.strip():
                    key, value = item.split(":")
                    field_dict[key.strip().strip("' ")] = float(value.strip())
            x = FIELD_AU_TO_VA * field_dict.get('X', 0.0)
            y = FIELD_AU_TO_VA * field_dict.get('Y', 0.0)
            z = FIELD_AU_TO_VA * field_dict.get('Z', 0.0)
            atoms.positions -= np.mean(atoms.positions, axis=0)
            atoms.info['external_field'] = np.array([x, y, z])
        except Exception as e:
            logging.warning(f"Failed to parse multipole_field: {atoms.info['multipole_field']}. Error: {e}")
            atoms.info['external_field'] = np.array([0.0, 0.0, 0.0])
    else:
        atoms.info['external_field'] = np.zeros(3)
    return atoms

In [7]:
import pandas as pd
import numpy as np
from tqdm import tqdm
from ase.io import read
# from mace.calculators import mace_anicc, mace_mp, mace_off, mace_omol, mace_polar
from mace.calculators import  mace_polar

df_refs = pd.read_csv("../Info/DatasetEval.csv", header=0)
df_energies = pd.read_csv("Molecule_Energies.csv", header=0)

# for a list of all foundational models: https://mace-docs.readthedocs.io/en/latest/guide/foundation_models.html
method = "polar-1-l"
calc = mace_polar(model= method)

df_energies[method] = pd.NA

for _, row in tqdm(df_refs.head(50).iterrows(), total=len(df_refs)):
        identifier = row['Reaction']
        reactions = str(row["Stoichiometry"]).split(",")
        num_species = len(reactions) // 2

        num_atoms = 0
        for i in range(num_species):
            stoi = float(reactions[2*i])
            specy = reactions[2*i+1].strip()
            atoms = process_atoms(read(f'../xyz_files/{specy}.xyz'))
            atoms.calc = calc
            energy = atoms.get_potential_energy()
            df_energies.loc[df_energies["Unnamed: 0"] == specy, method] = energy
            
df_energies.to_csv("Molecule_Energies_New.csv", index=False)

Using MACE-Polar model for MACECalculator with C:\Users\91988\.cache\mace/MACEPOLAR1Lmodel


c:\Users\91988\anaconda3\envs\mace-polar\Lib\site-packages\mace\calculators\mace.py:226: UserWarning: Environment variable TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD detected, since the`weights_only` argument was not explicitly passed to `torch.load`, forcing weights_only=False.
  torch.load(f=model_path, map_location=device)
  1%|          | 50/8448 [16:37<46:31:30, 19.94s/it]
